# Walmart Store Sales Forecasting — DLinear

**ლოგირება:** Weights & Biases — პროექტი `walmart-sales-forecasting`, group `DLinear_Training`. ნეირონული ქსელისთვის wandb განსაკუთრებით მოსახერხებელია: ეპოქების loss-ის მრუდები ავტომატურად იხაზება.

**რა არის DLinear** (Zeng et al., *"Are Transformers Effective for Time Series Forecasting?"*, AAAI 2023):
სერია იშლება ორ კომპონენტად moving-average-ით — **trend** და **seasonal (remainder)**. თითოეულზე გამოიყენება **ერთი წრფივი შრე**, რომელიც ბოლო `seq_len` კვირიდან პირდაპირ `pred_len` კვირას პროგნოზირებს, შედეგები კი იკრიბება. ავტორებმა აჩვენეს, რომ ეს უმარტივესი მოდელი ხშირად Transformer-ებს (Informer, Autoformer, FEDformer) აჯობებს long-horizon forecasting-ში — ეს ჩვენი პროექტის ერთ-ერთი მთავარი კვლევითი კითხვაა.

Run-ების სტრუქტურა:
1. `DLinear_Preprocessing` — Date × (Store, Dept) მატრიცის აგება, გაწმენდა, ნორმალიზაცია
2. `DLinear_candidate_{i}` — seq_len / kernel_size / individual sweep, შეფასება გუნდის საერთო holdout-ზე WMAE-თი
3. `DLinear_Final_Pipeline` — საბოლოო მოდელი მთელ ისტორიაზე, 39-კვირიანი ჰორიზონტით; ინახება W&B Artifact-ად როგორც end-to-end pipeline, რომელიც პირდაპირ raw test-ზე ეშვება

> Runtime: GPU აჩქარებს, მაგრამ მოდელი იმდენად პატარაა, რომ CPU-ზეც რამდენიმე წუთში იტრენინგება.

In [12]:
!pip install -q wandb

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import wandb

wandb.login()  # API key: https://wandb.ai/authorize

WANDB_PROJECT = 'ML-Final'  # გუნდის საერთო პროექტი
GROUP = 'DLinear_Training'  # "ექსპერიმენტი" ამ არქიტექტურისთვის — ყველა run ამ ჯგუფში ერთიანდება
NB_VERSION = 'v1'  # ჩაიწერება run-ების config-ში, რომ wandb-ზე ვერსიები გაიფილტროს

In [15]:
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/ML Final/Data/Raw/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(BASE + 'train.csv')
test = pd.read_csv(BASE + 'test.csv')
features = pd.read_csv(BASE + 'features.csv')
stores = pd.read_csv(BASE + 'stores.csv')

for d in (train, test, features):
    d['Date'] = pd.to_datetime(d['Date'])

RAW_COLS = ['Store', 'Dept', 'Date', 'IsHoliday']  # ზუსტად ის სვეტები, რაც raw test.csv-შია
VAL_CUTOFF = '2012-08-17'  # იგივე time split, რაც გუნდმა EDA_Preprocessing-ში გამოიყენა


def wmae(y_true, y_pred, is_holiday):
    """Competition metric: MAE, სადაც სადღესასწაულო კვირებს წონა 5 აქვთ."""
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)


print(train.shape, test.shape, features.shape, stores.shape)

(421570, 5) (115064, 4) (8190, 12) (45, 3)


## Run 1 — Preprocessing

DLinear უნივარიატულ ფანჯრებზე მუშაობს, ამიტომ long ცხრილს ვშლით **Date × (Store, Dept)** მატრიცად:
- უარყოფითი sales → 0 (გუნდის EDA-ს გადაწყვეტილება)
- სერიები, რომლებიც გვიან იწყება ან გამოტოვებული კვირები აქვთ → 0-ით ივსება
- თითო სერია ცალკე სტანდარტიზდება (მასშტაბები ათასჯერ განსხვავდება Store-Dept წყვილებს შორის); std < 1 → 1, რომ მკვდარი სერიები არ „აფეთქდეს"

In [16]:
run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='DLinear_Preprocessing',
                 job_type='preprocessing',
                 config={'negative_sales_strategy': 'clip_to_zero',
                         'missing_strategy': 'fill_zero (late-starting / gappy series)',
                         'normalization': 'per-series standardization, std clipped to >= 1'})

train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)

wide = train.pivot_table(index='Date', columns=['Store', 'Dept'], values='Weekly_Sales')
wide = wide.sort_index()
wide.index.name = 'Date'

missing_pct = float(wide.isna().mean().mean())
wide = wide.fillna(0.0)

run.config['matrix_shape'] = str(wide.shape)
run.log({'missing_cell_pct': missing_pct, 'n_series': wide.shape[1], 'n_weeks': wide.shape[0]})
print('weeks x series:', wide.shape, '| missing cells:', f'{missing_pct:.1%}')
run.finish()

val_dates = wide.index[wide.index >= VAL_CUTOFF]
PRED_LEN_VAL = len(val_dates)
print('validation weeks:', PRED_LEN_VAL)

weeks x series: (143, 3331) | missing cells: 11.5%


missing_cell_pct,▁
n_series,▁
n_weeks,▁
missing_cell_pct,0.11497
n_series,3331
n_weeks,143


validation weeks: 11


## DLinear იმპლემენტაცია (PyTorch)

- `MovingAvg` — მოძრავი საშუალო კიდეების replicate-padding-ით (ორიგინალი paper-ის მიხედვით)
- `SeriesDecomp` — `x = seasonal + trend` დაშლა
- `DLinear` — თითო კომპონენტზე ერთი `Linear(seq_len -> pred_len)` დროის ღერძზე:
  - `individual=False`: ერთი საერთო წრფივი შრე ყველა სერიაზე (channel-shared)
  - `individual=True`: თითო Store-Dept სერიას თავისი წრფივი შრე აქვს (paper-ის ვარიანტი; ბევრად მეტი პარამეტრი)

In [17]:
import torch
import torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print('device:', DEVICE)


class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):  # x: (B, L, C)
        front = x[:, :1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, self.kernel_size // 2, 1)
        x_pad = torch.cat([front, x, end], dim=1)
        return self.avg(x_pad.permute(0, 2, 1)).permute(0, 2, 1)


class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)
        return x - trend, trend


class DLinear(nn.Module):
    """Zeng et al. (AAAI 2023): decomposition + ორი წრფივი შრე."""

    def __init__(self, seq_len, pred_len, kernel_size=25, individual=False, n_series=1):
        super().__init__()
        self.seq_len, self.pred_len = seq_len, pred_len
        self.individual, self.n_series = individual, n_series
        self.decomp = SeriesDecomp(kernel_size)
        if individual:
            self.lin_seasonal = nn.ModuleList(nn.Linear(seq_len, pred_len) for _ in range(n_series))
            self.lin_trend = nn.ModuleList(nn.Linear(seq_len, pred_len) for _ in range(n_series))
        else:
            self.lin_seasonal = nn.Linear(seq_len, pred_len)
            self.lin_trend = nn.Linear(seq_len, pred_len)

    def forward(self, x):  # (B, L, C) -> (B, pred_len, C)
        seasonal, trend = self.decomp(x)
        seasonal = seasonal.permute(0, 2, 1)  # (B, C, L)
        trend = trend.permute(0, 2, 1)
        if self.individual:
            out = torch.stack(
                [self.lin_seasonal[i](seasonal[:, i]) + self.lin_trend[i](trend[:, i])
                 for i in range(self.n_series)], dim=1)
        else:
            out = self.lin_seasonal(seasonal) + self.lin_trend(trend)
        return out.permute(0, 2, 1)

device: cuda


In [18]:
from torch.utils.data import DataLoader, TensorDataset


def make_windows(arr, seq_len, pred_len):
    """arr: (T, N) -> sliding windows X:(n, seq_len, N), Y:(n, pred_len, N)."""
    xs, ys = [], []
    for t in range(arr.shape[0] - seq_len - pred_len + 1):
        xs.append(arr[t:t + seq_len])
        ys.append(arr[t + seq_len:t + seq_len + pred_len])
    return np.stack(xs), np.stack(ys)


def train_dlinear(arr, seq_len, pred_len, kernel_size=25, individual=False,
                  epochs=40, batch_size=16, lr=1e-3, log_fn=None):
    X, Y = make_windows(arr, seq_len, pred_len)
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(Y, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    model = DLinear(seq_len, pred_len, kernel_size, individual, n_series=arr.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses = []
    for ep in range(epochs):
        model.train()
        tot = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            tot += loss.item() * len(xb)
        losses.append(tot / len(ds))
        if log_fn:
            log_fn(ep, losses[-1])
        if (ep + 1) % 10 == 0:
            print(f'  epoch {ep + 1:>3}: train MSE {losses[-1]:.4f}')
    return model, losses


def forecast_next(model, arr, seq_len):
    """ისტორიის ბოლო seq_len კვირიდან შემდეგი pred_len კვირის პროგნოზი: (pred_len, N)."""
    model.eval()
    x = torch.tensor(arr[-seq_len:][None], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return model(x).cpu().numpy()[0]


val_actual = train[train['Date'] >= VAL_CUTOFF][['Store', 'Dept', 'Date', 'IsHoliday', 'Weekly_Sales']]


def matrix_val_wmae(pred_matrix):
    """(PRED_LEN_VAL, N) პროგნოზის მატრიცა -> WMAE ვალიდაციის რეალურ სტრიქონებზე."""
    pred_df = pd.DataFrame(pred_matrix, index=val_dates, columns=wide.columns)
    pred_df.index.name = 'Date'
    long = pred_df.stack(level=[0, 1], future_stack=True).rename('pred').reset_index()
    m = val_actual.merge(long, on=['Store', 'Dept', 'Date'], how='left')
    m['pred'] = m['pred'].fillna(0).clip(lower=0)
    return wmae(m['Weekly_Sales'], m['pred'], m['IsHoliday'])

## Run 2 — Hyperparameter Search

შესადარებელი პარამეტრები:
- `seq_len` (lookback): 26 / 52 / 78 კვირა — 52 კვირა სრულ წლიურ სეზონურობას იჭერს
- `kernel_size`: 13 / 25 — რამდენად გლუვია trend კომპონენტი
- `individual`: საერთო წრფივი შრე vs თითო სერიაზე ცალკე

ვალიდაცია: მოდელი ტრენინგდება `< 2012-08-17` მონაცემზე და ერთი ნაბიჯით პროგნოზირებს holdout-ის ყველა კვირას (`pred_len = PRED_LEN_VAL`), WMAE კი ითვლება ვალიდაციის რეალურ (Store, Dept, Date) სტრიქონებზე — ზუსტად ისე, როგორც სხვა არქიტექტურებში, ამიტომ შედეგები პირდაპირ შედარებადია. თითო კანდიდატი ცალკე wandb run-ია, ეპოქების loss კი ავტომატურად იხაზება.

In [19]:
tr_wide = wide[wide.index < VAL_CUTOFF]
mu = tr_wide.values.mean(axis=0)
sigma = np.clip(tr_wide.values.std(axis=0), 1.0, None)
norm_tr = (tr_wide.values - mu) / sigma

# v1 გაშვების დასკვნები: seq=52 ოპტიმალურია (26 ცოტაა: 2219; 104-ზე windows არ ჰყოფნის: 2232);
# kernel-ის ეფექტი მცირეა; individual=True overfit-დება (1946 — ტრენინგის loss ყველაზე
# დაბალი, ვალიდაცია ცუდი); lr=0.01-მა grid-ის კიდეზე მოიგო (1804 -> 1680).
# v2: lr-ის მიმართულება ღრმავდება (grid-edge წესი) + მეტი ეპოქა:
CANDIDATES = [
    dict(seq_len=52, kernel_size=25, individual=False, lr=1e-2),              # v1 საუკეთესო — reference
    dict(seq_len=52, kernel_size=25, individual=False, lr=2e-2),
    dict(seq_len=52, kernel_size=25, individual=False, lr=3e-2),
    dict(seq_len=52, kernel_size=25, individual=False, lr=1e-2, epochs=80),
    dict(seq_len=78, kernel_size=25, individual=False, lr=1e-2),              # v1-ის მე-2 seq + ახალი lr
]

best_val, BEST_CFG = np.inf, None
for i, cfg in enumerate(CANDIDATES):
    run = wandb.init(project=WANDB_PROJECT, group=GROUP, name=f'DLinear_candidate_{i}',
                     job_type='hyperparam_search',
                     config={**cfg, 'pred_len': PRED_LEN_VAL,
                             'val_scheme': f'train < {VAL_CUTOFF}, predict {PRED_LEN_VAL} weeks ahead'})
    print(f'candidate {i}: {cfg}')
    model, losses = train_dlinear(norm_tr, pred_len=PRED_LEN_VAL, **cfg,
                                  log_fn=lambda ep, l: run.log({'epoch': ep, 'train_mse': l}))
    pred = forecast_next(model, norm_tr, cfg['seq_len']) * sigma + mu
    score = float(matrix_val_wmae(pred))
    run.summary['val_wmae'] = score
    run.finish()
    print(f'  -> val WMAE {score:,.2f}')
    if score < best_val:
        best_val, BEST_CFG = score, cfg

print('Best:', BEST_CFG, f'-> {best_val:,.2f}')

candidate 0: {'seq_len': 52, 'kernel_size': 25, 'individual': False, 'lr': 0.01}
  epoch  10: train MSE 0.6323
  epoch  20: train MSE 0.6204
  epoch  30: train MSE 0.6190
  epoch  40: train MSE 0.6199


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_mse,█▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,39
train_mse,0.61989
val_wmae,1685.55517


  -> val WMAE 1,685.56


candidate 1: {'seq_len': 52, 'kernel_size': 25, 'individual': False, 'lr': 0.02}
  epoch  10: train MSE 0.6236
  epoch  20: train MSE 0.6242
  epoch  30: train MSE 0.6247
  epoch  40: train MSE 0.6252


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_mse,█▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,39
train_mse,0.62518
val_wmae,1677.80872


...

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇████
train_mse,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,79
train_mse,0.62073
val_wmae,1730.61701


  -> val WMAE 1,730.62


candidate 4: {'seq_len': 78, 'kernel_size': 25, 'individual': False, 'lr': 0.01}
  epoch  10: train MSE 0.6435
  epoch  20: train MSE 0.5903
  epoch  30: train MSE 0.5808
  epoch  40: train MSE 0.5796


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_mse,█▅▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,39
train_mse,0.57961
val_wmae,1602.68659


  -> val WMAE 1,602.69
Best: {'seq_len': 78, 'kernel_size': 25, 'individual': False, 'lr': 0.01} -> 1,602.69


## Run 3 — Final Pipeline

საბოლოო მოდელი მთელ ისტორიაზე (2010-02-05 … 2012-10-26) ტრენინგდება და ერთბაშად პროგნოზირებს Kaggle test-ის **39 კვირას** (2012-11-02 … 2013-07-26).

„Pipeline"-ის მოთხოვნა აქ ასე სრულდება: `DLinearForecastPipeline` კლასი შიგნით ინახავს მზა პროგნოზების ცხრილს + fallback საშუალოებს (train-ში უნახავი Store-Dept წყვილებისთვის), ხოლო `predict()` **raw** test dataframe-ს (`Store, Dept, Date, IsHoliday`) იღებს და პროგნოზებს პირდაპირ აბრუნებს — preprocessing არ სჭირდება. მთელი ობიექტი `cloudpickle`-ით ინახება W&B Artifact-ად.

In [20]:
import cloudpickle

test_dates = pd.DatetimeIndex(np.sort(test['Date'].unique()), name='Date')
PRED_LEN_TEST = len(test_dates)
assert (test_dates[0] - wide.index.max()).days == 7, 'test must start right after train'
print('test horizon:', PRED_LEN_TEST, 'weeks')


class DLinearForecastPipeline:
    """End-to-end pipeline: raw test rows (Store, Dept, Date, IsHoliday) -> predictions.

    ინახავს DLinear-ის მზა პროგნოზების ცხრილს + ისტორიულ fallback საშუალოებს,
    ამიტომ ჩატვირთვის შემდეგ predict() პირდაპირ დაუმუშავებელ test set-ზე მუშაობს.
    """

    def __init__(self, forecast_long, sd_mean, dept_mean, global_mean):
        self.forecast_long = forecast_long
        self.sd_mean = sd_mean
        self.dept_mean = dept_mean
        self.global_mean = global_mean

    def predict(self, model_input):
        df = model_input.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        out = (df.merge(self.forecast_long, on=['Store', 'Dept', 'Date'], how='left')
                 .merge(self.sd_mean, on=['Store', 'Dept'], how='left')
                 .merge(self.dept_mean, on='Dept', how='left'))
        pred = (out['pred'].fillna(out['SD_Mean'])
                           .fillna(out['Dept_Mean'])
                           .fillna(self.global_mean))
        return pred.clip(lower=0).values


run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='DLinear_Final_Pipeline',
                 job_type='final_pipeline',
                 config={**BEST_CFG, 'pred_len': PRED_LEN_TEST})
run.summary['val_wmae'] = best_val  # შეფასება hyperparam search-ის holdout-იდან

mu_f = wide.values.mean(axis=0)
sigma_f = np.clip(wide.values.std(axis=0), 1.0, None)
norm_full = (wide.values - mu_f) / sigma_f

final_model, losses = train_dlinear(norm_full, pred_len=PRED_LEN_TEST, **BEST_CFG,
                                    log_fn=lambda ep, l: run.log({'epoch': ep, 'train_mse': l}))

pred = forecast_next(final_model, norm_full, BEST_CFG['seq_len']) * sigma_f + mu_f
pred = np.clip(pred, 0, None)

fc = pd.DataFrame(pred, index=test_dates, columns=wide.columns)
forecast_long = fc.stack(level=[0, 1], future_stack=True).rename('pred').reset_index()

sd_mean = (train.groupby(['Store', 'Dept'])['Weekly_Sales']
                .mean().rename('SD_Mean').reset_index())
dept_mean = (train.groupby('Dept')['Weekly_Sales']
                  .mean().rename('Dept_Mean').reset_index())

pipeline = DLinearForecastPipeline(forecast_long, sd_mean, dept_mean,
                                   float(train['Weekly_Sales'].mean()))

with open('dlinear_pipeline.pkl', 'wb') as f:
    cloudpickle.dump(pipeline, f)
torch.save(final_model.state_dict(), 'dlinear_state.pt')

artifact = wandb.Artifact('dlinear_pipeline', type='model',
                          description='DLinear end-to-end pipeline: raw test rows -> predictions',
                          metadata={'val_wmae': float(best_val), **BEST_CFG})
artifact.add_file('dlinear_pipeline.pkl')
artifact.add_file('dlinear_state.pt')
run.log_artifact(artifact)

# pipeline პირდაპირ RAW test set-ზე
test_pred = pipeline.predict(test[RAW_COLS])
submission = pd.DataFrame({
    'Id': (test['Store'].astype(str) + '_' + test['Dept'].astype(str)
           + '_' + test['Date'].dt.strftime('%Y-%m-%d')),
    'Weekly_Sales': test_pred,
})
submission.to_csv('submission_dlinear.csv', index=False)
sub_art = wandb.Artifact('dlinear_submission', type='submission')
sub_art.add_file('submission_dlinear.csv')
run.log_artifact(sub_art)

val_score = best_val
run.finish()
submission.head()

test horizon: 39 weeks


  epoch  10: train MSE 0.6207
  epoch  20: train MSE 0.5566
  epoch  30: train MSE 0.5396
  epoch  40: train MSE 0.5344


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_mse,█▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,39
train_mse,0.53441
val_wmae,1602.68659


,Id,Weekly_Sales
0,1_1_2012-11-02,33420.664386
1,1_1_2012-11-09,24393.368851
2,1_1_2012-11-16,21480.702701
3,1_1_2012-11-23,22876.366443
4,1_1_2012-11-30,24222.323519


In [21]:
# ვინახავთ საუკეთესო pipeline-ის artifact-ის სახელს Drive-ზე, რომ model_inference.ipynb-მ
# ყველა არქიტექტურა შეადაროს და საუკეთესო W&B Model Registry-ში დაალინკოს.
import json, os

reg_path = '/content/drive/MyDrive/ML Final/best_runs.json'
info = json.load(open(reg_path)) if os.path.exists(reg_path) else {}
info['DLinear'] = {
    'artifact': f'{WANDB_PROJECT}/dlinear_pipeline:latest',
    'val_wmae': float(val_score),
}
with open(reg_path, 'w') as f:
    json.dump(info, f, indent=2)
info

{'XGBoost': {'artifact': 'ML-Final/xgboost_pipeline:latest',
  'val_wmae': 1451.361784955096},
 'LightGBM': {'artifact': 'ML-Final/lightgbm_pipeline:latest',
  'val_wmae': 1508.4106169389534},
 'DLinear': {'artifact': 'ML-Final/dlinear_pipeline:latest',
  'val_wmae': 1602.6865936618267}}

In [22]:
# sanity check: pipeline W&B artifact-იდან ჩამოვტვირთოთ და raw test-ზე გავუშვათ
import cloudpickle, os

api = wandb.Api()
art = api.artifact(f'{WANDB_PROJECT}/dlinear_pipeline:latest')
art_dir = art.download()
with open(os.path.join(art_dir, 'dlinear_pipeline.pkl'), 'rb') as f:
    loaded = cloudpickle.load(f)
print(loaded.predict(test[RAW_COLS].head())[:5])

wandb:   2 of 2 files downloaded.  


[33420.66438615 24393.36885107 21480.70270073 22876.3664434
 24222.32351947]


## შემდეგი ნაბიჯი

- artifact-ის სახელი და `val_wmae` ჩაიწერა `best_runs.json`-ში — `model_inference.ipynb` არქიტექტურებს შეადარებს და საუკეთესოს W&B Model Registry-ში დაალინკავს (`art.link('wandb-registry-model/walmart-best-model')` ან UI-დან), inference კი მოდელს პირდაპირ registry-იდან ჩამოტვირთავს.
- README-სთვის ჩაინიშნეთ: საუკეთესო კონფიგურაცია (`seq_len`, `kernel_size`, `individual`), val WMAE და შედარება tree მოდელებთან.
- დაკვირვება report-ისთვის: DLinear-ს **არ აქვს** ეგზოგენური ფიჩერები (MarkDown, CPI, holidays) — მხოლოდ ისტორიას ხედავს. თუ tree მოდელები ჯობნიან, ეს სავარაუდოდ სწორედ holiday/markdown ინფორმაციის დამსახურებაა; ეს კარგი სადისკუსიო პუნქტია პრეზენტაციისთვის.